In [44]:
import asyncio
import os
import re
import uuid
from typing import List, Optional

from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from openai import OpenAI
from neo4j import AsyncGraphDatabase
from llama_cloud_services.extract import (
    LlamaExtract,
    ExtractConfig,
    ExtractMode,
    SourceText,
)
from llama_cloud_services.parse import LlamaParse
from neo4j import GraphDatabase
from dotenv import load_dotenv
from pathlib import Path

env_path = Path("/home/luis/Documents/FGV/Laboratory/document-graph/server/.env")
load_dotenv(dotenv_path=env_path)

True

In [45]:
classification_prompt = """You are a legal document classification assistant.
Your task is to identify the most likely contract type based on the content of the first 10 pages of a contract.
Instructions:
Read the contract excerpt below.
Review the list of possible contract types.
Choose the single most appropriate contract type from the list.
Justify your classification briefly, based only on the information in the excerpt.

Contract Excerpt:
{contract_text}
Possible Contract Types:
{contract_type_list}

Output Format:
<Reason>brief_justification</Reason>
<ContractType>chosen_type_from_list</ContractType>
"""


In [46]:

def extract_reason_and_contract_type(text: str) -> dict:
    reason_match = re.search(r"<Reason>(.*?)</Reason>", text, re.DOTALL | re.IGNORECASE)
    type_match = re.search(r"<ContractType>(.*?)</ContractType>", text, re.DOTALL | re.IGNORECASE)
    reason = reason_match.group(1).strip() if reason_match else ""
    contract_type = type_match.group(1).strip() if type_match else ""
    return {"reason": reason, "contract_type": contract_type}

def classify_contract(openai_client, contract_text: str, contract_types: list[str]) -> dict:
    prompt = classification_prompt.format(
        contract_text=contract_text,
        contract_type_list=", ".join(contract_types),
    )
    response = openai_client.responses.create(
        input=[{"role": "user", "content": prompt}],
        model="gpt-4o-mini",
        store=False,
    )
    return extract_reason_and_contract_type(response.output[0].content[0].text)

In [47]:
class Location(BaseModel):
    country: Optional[str] = Field(None, description="Country")
    state: Optional[str] = Field(None, description="State or province")
    address: Optional[str] = Field(None, description="Street address or city")


class Party(BaseModel):
    name: str = Field(description="Party name")
    location: Optional[Location] = Field(None, description="Party location details")


class BaseContract(BaseModel):
    parties: Optional[List[Party]] = Field(None, description="All contracting parties")
    agreement_date: Optional[str] = Field(None, description="Contract signing date. Use YYYY-MM-DD")
    effective_date: Optional[str] = Field(None, description="When contract becomes effective. Use YYYY-MM-DD")
    expiration_date: Optional[str] = Field(None, description="Contract expiration date. Use YYYY-MM-DD")
    governing_law: Optional[str] = Field(None, description="Governing jurisdiction")
    termination_for_convenience: Optional[bool] = Field(None, description="Can terminate without cause")
    anti_assignment: Optional[bool] = Field(None, description="Restricts assignment to third parties")
    cap_on_liability: Optional[str] = Field(None, description="Liability limit amount")


class AffiliateAgreement(BaseContract):
    exclusivity: Optional[str] = Field(None, description="Exclusive territory or market rights")
    non_compete: Optional[str] = Field(None, description="Non-compete restrictions")
    revenue_profit_sharing: Optional[str] = Field(None, description="Commission or revenue split")
    minimum_commitment: Optional[str] = Field(None, description="Minimum sales targets")


class CoBrandingAgreement(BaseContract):
    exclusivity: Optional[str] = Field(None, description="Exclusive co-branding rights")
    ip_ownership_assignment: Optional[str] = Field(None, description="IP ownership allocation")
    license_grant: Optional[str] = Field(None, description="Brand/trademark licenses")
    revenue_profit_sharing: Optional[str] = Field(None, description="Revenue sharing terms")


class DevelopmentAgreement(BaseContract):
    deliverables: Optional[str] = Field(None, description="Project deliverables")
    milestones: Optional[str] = Field(None, description="Milestones and schedule")
    ip_ownership_assignment: Optional[str] = Field(None, description="IP ownership allocation")
    acceptance_criteria: Optional[str] = Field(None, description="Acceptance/testing conditions")



In [48]:
mapping = {
    "Affiliate_Agreements": AffiliateAgreement,
    "Co_Branding": CoBrandingAgreement,
    "Development": DevelopmentAgreement,
}


In [49]:
import_query = """
WITH $contract AS contract
MERGE (c:Contract {path: $path})
SET c += apoc.map.clean(contract, ["parties", "agreement_date", "effective_date", "expiration_date"], [])
SET c.agreement_date = CASE WHEN contract.agreement_date IS NULL OR contract.agreement_date = "" THEN NULL ELSE date(contract.agreement_date) END,
    c.effective_date = CASE WHEN contract.effective_date IS NULL OR contract.effective_date = "" THEN NULL ELSE date(contract.effective_date) END,
    c.expiration_date = CASE WHEN contract.expiration_date IS NULL OR contract.expiration_date = "" THEN NULL ELSE date(contract.expiration_date) END
WITH c, contract
UNWIND coalesce(contract.parties, []) AS party
MERGE (p:Party {name: party.name})
MERGE (c)-[:HAS_PARTY]->(p)
WITH p, party
WHERE party.location IS NOT NULL
MERGE (l:Location {
  address: coalesce(party.location.address, ""),
  state: coalesce(party.location.state, ""),
  country: coalesce(party.location.country, "")
})
MERGE (p)-[:HAS_LOCATION]->(l)
"""

In [50]:
def main():
    llama_api_key = os.environ["LLAMA_CLOUD_API_KEY"]
    openai_api_key = os.environ["OPENAI_API_KEY"]

    neo4j_uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
    neo4j_username = os.getenv("NEO4J_USERNAME", "neo4j")
    neo4j_password = os.getenv("NEO4J_PASSWORD", "neo4j_password")
    neo4j_database = os.getenv("NEO4J_DATABASE", "neo4j")

    pdf_path = "/home/luis/Documents/FGV/Laboratory/document-graph/infra/CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf"
    contract_types = ["Affiliate_Agreements", "Co_Branding", "Development"]

    parser = LlamaParse(
        api_key=llama_api_key,
        parse_mode="parse_page_without_llm",
    )
    results = parser.parse(pdf_path) 

    openai_client = OpenAI(api_key=openai_api_key)
    file_content = " ".join([el.text for el in results.pages[:10]])

    classification = classify_contract(openai_client, file_content, contract_types)
    contract_type = classification["contract_type"]
    if contract_type not in mapping:
        raise ValueError(f"Unsupported contract type: {contract_type}")

    extractor = LlamaExtract(api_key=llama_api_key)
    agent = extractor.create_agent(
        name=f"extraction_workflow_import_{uuid.uuid4()}",
        data_schema=mapping[contract_type],
        config=ExtractConfig(extraction_mode=ExtractMode.BALANCED),
    )

    result = agent.extract( 
        files=SourceText(
            text_content=" ".join([el.text for el in results.pages]),
            filename=pdf_path,
        ),
    )

    contract_payload = result.data if isinstance(result.data, dict) else result.data.model_dump()

    neo4j_driver = GraphDatabase.driver(
        neo4j_uri,
        auth=(neo4j_username, neo4j_password),
    )
    try:
        response = neo4j_driver.execute_query( 
            import_query,
            contract=contract_payload,
            path=pdf_path,
            database_=neo4j_database,
        )
        print("Classification:", classification)
        print("Neo4j counters:", response.summary.counters)
    finally:
        neo4j_driver.close()


main()

Started parsing the file under job_id 11ad87bd-cabc-4af8-888b-831ce2ae2196


Extracting files: 100%|██████████| 1/1 [00:36<00:00, 36.33s/it]


Classification: {'reason': 'This document is classified as a Marketing Affiliate Agreement. It outlines the responsibilities and rights of the parties involved in the marketing and support of specific technology products. The focus is on the marketing efforts of one company (the Marketing Affiliate) on behalf of another (the Company), which is typical of affiliate agreements where one party promotes the goods or services of another in exchange for fees or commissions.', 'contract_type': 'Affiliate_Agreements'}
Neo4j counters: SummaryCounters{labels_added: 5, relationships_created: 4, nodes_created: 5, properties_set: 17, contains_updates: True, contains_system_updates: False}
